# Phase 7: TOPSIS Multi-Criteria Ranking

By now we have six models and six numbers for each of them. Some models are the most accurate, some are the fastest, and some use the least energy. No single model wins everything, so the question is **which one should a cloud provider actually deploy?** TOPSIS is a standard method for answering exactly that kind of question.

**The idea in one sentence:** imagine a perfect model that has the best value of every criterion (the "ideal best") and a terrible model that has the worst of every criterion (the "ideal worst"). Then rank the real models by how close they are to the perfect one and how far from the terrible one.

**Steps** (each one is shown below so anyone can check the arithmetic):
1. **Decision matrix:** one row per model, one column per criterion.
2. **Normalise:** divide each column by its length (√ of the sum of squares), so seconds, grams and R² all end up on a comparable 0–1 scale.
3. **Weight:** multiply each column by how important it is.
4. **Ideal best and worst:** for "higher is better" criteria (R²) the best is the column maximum. For "lower is better" ones (errors, time, CO₂) the best is the minimum.
5. **Distances:** the straight-line distance of each model to the ideal best (*D⁺*) and to the ideal worst (*D⁻*).
6. **Closeness:** *C = D⁻ / (D⁺ + D⁻)*, between 0 and 1. Higher is better, and the model with the highest *C* ranks first.

**Inputs** (agreed by the team): the **tuned** version of every model.
- Accuracy (MAE, RMSE, MAPE, R²) comes from Phase 4, on the test set.
- Training time and CO₂ come from Phase 5, measured together in the same CodeCarbon run, so they describe the same training.

## 0. Setup

In [1]:
# Mount Google Drive (Colab) or use the local Drive folder (PC), then load our shared code.
import os, sys, importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Green_ML_Project'
except ImportError:
    PROJECT_ROOT = r'H:\My Drive\Green_ML_Project'   # running on the PC instead of Colab

os.environ['GREEN_ML_ROOT'] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts'))

import config as C
import utils as U
importlib.reload(C); importlib.reload(U)   # picks up any edits to the scripts without restarting

print('Project folder:', C.PROJECT_ROOT)
print('Folder found:  ', os.path.isdir(C.PROJECT_ROOT))

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Green_ML_Project
Folder found:   True


In [2]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
import model_definitions as M
importlib.reload(M)

pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda v: f'{v:.6g}')
log = U.get_logger('phase07_topsis_ranking')

RUN STARTED 2026-10-04 07:25:34 | phase07_topsis_ranking
python 3.13.15 | pandas 2.2.3 | numpy 2.1.3 | machine: Linux-6.6.122+-x86_64-with-glibc2.39
project root: /content/drive/MyDrive/Green_ML_Project


## 1. Decision matrix

In [3]:
phase3 = json.load(open(C.MODELS_DIR / 'phase3_results.json'))
phase4 = json.load(open(C.MODELS_DIR / 'phase4_results.json'))
phase5 = json.load(open(C.MODELS_DIR / 'phase5_results.json'))

order = M.TABULAR_MODELS + ['LSTM']
rows = []
for name in order:
    acc = phase4.get(name, phase3[name])          # Linear Regression has no tuning -> Phase 3 values
    energy = phase5[f'{name} | tuned']
    rows.append({'Model': name, 'MAE': acc['MAE'], 'RMSE': acc['RMSE'], 'MAPE': acc['MAPE'], 'R2': acc['R2'],
                 'train_time_sec': energy['train_time_sec'], 'co2_emissions_kg': energy['co2_emissions_kg']})
decision = pd.DataFrame(rows).set_index('Model')
decision.to_csv(C.TABLES_DIR / 'phase7_decision_matrix.csv')
U.log_df(log, decision, 'DECISION MATRIX (tuned models)')
U.save_table(decision.reset_index(), 'phase7_decision_matrix', 'TOPSIS decision matrix (tuned models)', float_fmt='{:.4g}')
decision

--- DECISION MATRIX (tuned models) ---
                       MAE    RMSE    MAPE       R2  train_time_sec  co2_emissions_kg
Model                                                                                
Linear Regression 0.903215  4.5899 18.2409 0.906246        0.253325       4.07064e-07
Decision Tree      1.05024 4.35614 28.0678 0.915553         2.04198        2.9542e-06
Random Forest     0.696579 3.74933 14.0398 0.937441         194.818       0.000349457
XGBoost           0.697399 3.71514 13.4264 0.938577         5.92008       8.30255e-06
LightGBM          0.664048 3.65074  12.417 0.940688         7.81968       1.26522e-05
LSTM              0.923798 4.49392 19.7413 0.910126         158.908       0.000280536


,MAE,RMSE,MAPE,R2,train_time_sec,co2_emissions_kg
Model,,,,,,
Linear Regression,0.903215,4.5899,18.2409,0.906246,0.253325,4.07064e-07
Decision Tree,1.05024,4.35614,28.0678,0.915553,2.04198,2.9542e-06
Random Forest,0.696579,3.74933,14.0398,0.937441,194.818,0.000349457
XGBoost,0.697399,3.71514,13.4264,0.938577,5.92008,8.30255e-06
LightGBM,0.664048,3.65074,12.417,0.940688,7.81968,1.26522e-05
LSTM,0.923798,4.49392,19.7413,0.910126,158.908,0.000280536


## 2. Weights and why we chose them

| Criterion | Weight | Direction |
|---|---|---|
| MAE | 0.15 | lower is better |
| RMSE | 0.15 | lower is better |
| MAPE | 0.15 | lower is better |
| R² | 0.20 | higher is better |
| Training time | 0.15 | lower is better |
| CO₂ emissions | 0.20 | lower is better |
| **Total** | **1.00** | |

**Why this split:**
- **Accuracy still carries most of the weight (0.65 in total).** A load forecaster exists to help the data centre provision the right amount of capacity. A poor forecast leads to over-provisioning, which wastes far more energy than training the model ever does, or to under-provisioning, which breaks service agreements. A "green" model that forecasts badly isn't green in practice.
- **R² gets slightly more (0.20) than each error measure (0.15).** MAE, RMSE and MAPE each look at error from a different angle: average size, big misses, and relative size. R² summarises how much of the workload's behaviour the model captures overall, independent of scale.
- **CO₂ gets 0.20, the joint-highest single weight.** It's the criterion this paper is about, and the one most often ignored in the literature we reviewed.
- **Training time gets 0.15.** It's an operational cost in its own right: workload models need regular retraining as workloads drift, and a slow model also ties up machines that could be serving customers.

**Two honest caveats**, both tested in Phase 8:
1. The three error measures tend to move together, so accuracy may count for a bit more in practice than its 0.65 suggests.
2. On Colab, training time and CO₂ are closely linked, because CodeCarbon estimates energy from time (see Phase 5). Together they act like one "cost" criterion with a weight of 0.35.

Phase 8 re-runs the ranking with very different weightings to see whether the recommendation survives.

In [4]:
weights = C.TOPSIS_PRIMARY_WEIGHTS
benefit = C.TOPSIS_CRITERIA
assert abs(sum(weights.values()) - 1) < 1e-9
log.info(f'Weights: {weights}')
log.info(f'Higher is better: {[c for c, b in benefit.items() if b]} | lower is better: {[c for c, b in benefit.items() if not b]}')
log.info(f'Correlation between train time and CO2 across models: {decision.train_time_sec.corr(decision.co2_emissions_kg):.3f}')

Weights: {'MAE': 0.15, 'RMSE': 0.15, 'MAPE': 0.15, 'R2': 0.2, 'train_time_sec': 0.15, 'co2_emissions_kg': 0.2}
Higher is better: ['R2'] | lower is better: ['MAE', 'RMSE', 'MAPE', 'train_time_sec', 'co2_emissions_kg']
Correlation between train time and CO2 across models: 1.000


## 3. TOPSIS step by step

In [5]:
res = U.topsis(decision, weights, benefit)

print('Step 2: normalised matrix')
display(res['normalized'])
print('Step 3: weighted normalised matrix')
display(res['weighted'])
print('Step 4: ideal best and ideal worst')
ideal = pd.DataFrame({'Ideal best (A+)': res['ideal_best'], 'Ideal worst (A-)': res['ideal_worst']}).T
display(ideal)

U.log_df(log, res['normalized'], 'STEP 2: NORMALISED MATRIX')
U.log_df(log, res['weighted'], 'STEP 3: WEIGHTED NORMALISED MATRIX')
U.log_df(log, ideal, 'STEP 4: IDEAL BEST / WORST')
res['weighted'].to_csv(C.TABLES_DIR / 'phase7_weighted_matrix.csv')
ideal.to_csv(C.TABLES_DIR / 'phase7_ideal_solutions.csv')

Step 2: normalised matrix


,MAE,RMSE,MAPE,R2,train_time_sec,co2_emissions_kg
Model,,,,,,
Linear Regression,0.441526,0.455752,0.40368,0.400021,0.00100683,0.000907823
Decision Tree,0.513399,0.432541,0.621155,0.404129,0.00811573,0.00658839
Random Forest,0.340514,0.372287,0.310707,0.413791,0.774294,0.77935
XGBoost,0.340915,0.368893,0.297133,0.414292,0.0235291,0.0185161
LightGBM,0.324611,0.362499,0.274796,0.415224,0.0310789,0.0282166
LSTM,0.451587,0.446222,0.436886,0.401734,0.631571,0.625644


Step 3: weighted normalised matrix


,MAE,RMSE,MAPE,R2,train_time_sec,co2_emissions_kg
Model,,,,,,
Linear Regression,0.0662288,0.0683627,0.0605521,0.0800043,0.000151024,0.000181565
Decision Tree,0.0770099,0.0648812,0.0931732,0.0808259,0.00121736,0.00131768
Random Forest,0.0510771,0.0558431,0.0466061,0.0827582,0.116144,0.15587
XGBoost,0.0511373,0.0553339,0.0445699,0.0828585,0.00352936,0.00370322
LightGBM,0.0486917,0.0543748,0.0412193,0.0830448,0.00466184,0.00564331
LSTM,0.0677381,0.0669332,0.0655329,0.0803468,0.0947356,0.125129


Step 4: ideal best and ideal worst


,MAE,RMSE,MAPE,R2,train_time_sec,co2_emissions_kg
Ideal best (A+),0.0486917,0.0543748,0.0412193,0.0830448,0.000151024,0.000181565
Ideal worst (A-),0.0770099,0.0683627,0.0931732,0.0800043,0.116144,0.15587


--- STEP 2: NORMALISED MATRIX ---
                       MAE     RMSE     MAPE       R2  train_time_sec  co2_emissions_kg
Model                                                                                  
Linear Regression 0.441526 0.455752  0.40368 0.400021      0.00100683       0.000907823
Decision Tree     0.513399 0.432541 0.621155 0.404129      0.00811573        0.00658839
Random Forest     0.340514 0.372287 0.310707 0.413791        0.774294           0.77935
XGBoost           0.340915 0.368893 0.297133 0.414292       0.0235291         0.0185161
LightGBM          0.324611 0.362499 0.274796 0.415224       0.0310789         0.0282166
LSTM              0.451587 0.446222 0.436886 0.401734        0.631571          0.625644
--- STEP 3: WEIGHTED NORMALISED MATRIX ---
                        MAE      RMSE      MAPE        R2  train_time_sec  co2_emissions_kg
Model                                                                                      
Linear Regression 0.0662288 0.06836

## 4. Final ranking

In [6]:
ranking = pd.DataFrame({
    'Model': decision.index,
    'D+ (distance to best)': res['d_best'].values,
    'D- (distance to worst)': res['d_worst'].values,
    'Closeness C': res['closeness'].values,
    'Rank': res['rank'].values,
}).sort_values('Rank').reset_index(drop=True)
ranking.to_csv(C.TABLES_DIR / 'phase7_topsis_ranking.csv', index=False)
U.log_df(log, ranking, 'TOPSIS RANKING (primary weights)')
U.save_table(ranking, 'phase7_topsis_ranking', 'TOPSIS ranking with the primary weights', float_fmt='{:.4f}')

winner = ranking.iloc[0]
log.info(f'Recommended model: {winner.Model} (C = {winner["Closeness C"]:.4f}); '
         f'runner-up: {ranking.iloc[1].Model} (C = {ranking.iloc[1]["Closeness C"]:.4f})')
ranking

--- TOPSIS RANKING (primary weights) ---
               Model  D+ (distance to best)  D- (distance to worst)  Closeness C  Rank
0            XGBoost             0.00647897                0.197602     0.968253     1
1           LightGBM             0.00708365                0.196729     0.965244     2
2  Linear Regression              0.0297693                0.197164     0.868819     3
3      Decision Tree              0.0601569                0.192633     0.762028     4
4               LSTM                0.16024               0.0474917      0.22862     5
5      Random Forest               0.194243               0.0548209     0.220108     6
Recommended model: XGBoost (C = 0.9683); runner-up: LightGBM (C = 0.9652)


,Model,D+ (distance to best),D- (distance to worst),Closeness C,Rank
0,XGBoost,0.00647897,0.197602,0.968253,1
1,LightGBM,0.00708365,0.196729,0.965244,2
2,Linear Regression,0.0297693,0.197164,0.868819,3
3,Decision Tree,0.0601569,0.192633,0.762028,4
4,LSTM,0.16024,0.0474917,0.22862,5
5,Random Forest,0.194243,0.0548209,0.220108,6


In [7]:
# Category winners table (for the paper): who is best on each single criterion vs who TOPSIS recommends
winners = []
for c, higher in benefit.items():
    best = decision[c].idxmax() if higher else decision[c].idxmin()
    winners.append({'Criterion': c, 'Best model': best, 'Value': decision.loc[best, c],
                    'TOPSIS winner value': decision.loc[winner.Model, c]})
winners.append({'Criterion': 'TOPSIS closeness (overall)', 'Best model': winner.Model,
                'Value': winner['Closeness C'], 'TOPSIS winner value': winner['Closeness C']})
winners = pd.DataFrame(winners)
U.log_df(log, winners, 'CATEGORY WINNERS')
U.save_table(winners, 'phase7_category_winners', 'Best model per criterion vs the TOPSIS recommendation', float_fmt='{:.4g}')
log.info('PHASE 7 COMPLETE')
winners

--- CATEGORY WINNERS ---
                    Criterion         Best model       Value  TOPSIS winner value
0                         MAE           LightGBM    0.664048             0.697399
1                        RMSE           LightGBM     3.65074              3.71514
2                        MAPE           LightGBM      12.417              13.4264
3                          R2           LightGBM    0.940688             0.938577
4              train_time_sec  Linear Regression    0.253325              5.92008
5            co2_emissions_kg  Linear Regression 4.07064e-07          8.30255e-06
6  TOPSIS closeness (overall)            XGBoost    0.968253             0.968253
PHASE 7 COMPLETE


,Criterion,Best model,Value,TOPSIS winner value
0,MAE,LightGBM,0.664048,0.697399
1,RMSE,LightGBM,3.65074,3.71514
2,MAPE,LightGBM,12.417,13.4264
3,R2,LightGBM,0.940688,0.938577
4,train_time_sec,Linear Regression,0.253325,5.92008
5,co2_emissions_kg,Linear Regression,4.07064e-07,8.30255e-06
6,TOPSIS closeness (overall),XGBoost,0.968253,0.968253


## Done: what to check

- Does the recommended model make sense when you look at the decision matrix? TOPSIS rewards balance, so the winner usually isn't the best at any single thing but is good at everything.
- How far apart are the closeness scores? A big gap between ranks 1 and 2 means a clear recommendation. A tiny gap means the choice depends on the weights, which is exactly what Phase 8 tests.

Next: `08_sensitivity_analysis.ipynb`.